# B3 — mixed displacement-stress formulation: RELATIVE constitutive residual (attempt 4)

**Follow-up paper, Professor Rabczuk's item 1.** Three attempts so far have all failed with the SAME signature -- `constitutive` oscillating with no downward trend:
- Attempt 1 (`p_output_scale=500.0`): failed across the full 50,000 iterations.
- Attempt 2 (`p_output_scale=250.0`, recalibrated via `calibrate_p_output_scale.py`): interrupted at 14,000/50,000 once the identical oscillation appeared.
- Attempt 3 (two-stage curriculum, `--pretrain_iters 10000`): u pretrained cleanly via the proven energy objective (stage 1 looked healthy), but `constitutive` STILL oscillated with no trend after the stage-2 switch (interrupted at 29,500/50,000; range ~24,000-75,000 throughout, `equilibrium` behaved fine).

**What's different this time**: a direct CPU measurement of the TRUE |P| field across the real domain (real sampled E/nu/phi, same methodology as `calibrate_p_output_scale.py`) found it is extremely heavy-tailed: median |P|_F ~99, p90 ~332, p99 ~801 (8x median), max ~2599 (**26x** the median) -- almost certainly the groove region, where strain concentrates. The old constitutive residual is a plain absolute squared error, which weights every quadrature point's error equally regardless of local stress scale -- so the handful of high-|P| (likely groove) points can dominate the loss/gradient far out of proportion to their share of the domain. This is a real, measured candidate explanation that fits all three previous failures (no implementation bug was ever found despite three full re-audits -- `test_mixed_stress_B3.py`'s 6 CPU checks, now including one for this new path, all pass).

**The fix**: `mixed_stress_loss_B3(..., relative_constitutive=True, constitutive_eps=400.0)` divides each quadrature point's constitutive residual by `(constitutive_eps + |P_true|_F^2)` before volume-weighting -- a genuinely relative error that can no longer be dominated by a few high-stress points purely because of their absolute magnitude. `constitutive_eps=400.0` (~20²) floors the denominator at the measured 10th-percentile |P|_F (~20.5) so very-low-stress points (measured min |P|_F ~0.21) aren't divided by a near-zero number and don't swing to dominate the opposite way.

**Re-measured `w_equilibrium`, not reused**: under the new relative loss, `constitutive` and `equilibrium` are naturally within about one order of magnitude of each other at initialization (measured on the real production-size model, 3 seeds: constitutive ~10-20, equilibrium ~45-78) -- nothing like the 28,000x imbalance that made the old `w_equilibrium=28000` necessary for the absolute loss. **This run uses `w_equilibrium=1.0`.**

**No curriculum this time** (`--pretrain_iters 0`) -- isolating this one change (the residual itself) from the previous one (training order), so if it works we know which fix mattered, and if it fails the same way again, that would be strong evidence the problem is something else entirely (architecture capacity, learning rate, or the groove region's intrinsic difficulty needing its own treatment, e.g. local mesh/integration refinement the way `evaluate_B3_region_local_refine.py` already does for the energy-based baseline).

**Watch `constitutive` closely**: it is now O(1), not O(10,000) -- a real downward trend should be visible as a clear decrease from its initial ~10-80 range toward something much smaller, not just a different oscillation range. If it still shows no trend, interrupt and report -- this is now the fourth independent hypothesis being tested.

**Writes to a NEW output directory** (`b3_mixed_stress_relative`), keeping every previous attempt's checkpoints/logs separately inspectable.

In [ ]:
# =====================================================================
#  CELL -- B3 mixed displacement-stress: attempt 4, RELATIVE constitutive
#  residual (no curriculum this time -- isolating the one change), then
#  evaluation against the held-out FEM dataset.
#  See the markdown cell above for the full diagnosis and why this
#  specific change was made.
# =====================================================================
import os
os.environ['JAX_PLATFORMS'] = 'cpu'

import subprocess
import sys
import time

_started = time.time()


def run(cmd):
    print('$', ' '.join(str(c) for c in cmd), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    p.wait()
    if p.returncode != 0:
        raise subprocess.CalledProcessError(p.returncode, cmd)


from google.colab import drive
if os.path.isdir('/content/drive/MyDrive'):
    print('Drive already mounted at /content/drive.')
else:
    drive.mount('/content/drive')

REPO = '/content/OMAR'
if not os.path.isdir(REPO):
    run(['git', 'clone', '-b', 'claude/claude-code-question-d307wp',
         'https://github.com/SUHIBAMRO/OMAR.git', REPO])
else:
    run(['git', '-C', REPO, 'fetch', 'origin', 'claude/claude-code-question-d307wp'])
    run(['git', '-C', REPO, 'checkout', 'claude/claude-code-question-d307wp'])
    run(['git', '-C', REPO, 'reset', '--hard', 'origin/claude/claude-code-question-d307wp'])

run([sys.executable, '-m', 'pip', 'install', '-q', 'pyvista<0.49'])
run([sys.executable, '-m', 'pip', 'install', '-q', 'torch-fem'])

WORK = f'{REPO}/Practical_Examples'
os.chdir(WORK)
sys.path.insert(0, WORK)

for _mod_name in list(sys.modules):
    if (_mod_name == 'omar_pfem' or _mod_name.startswith('omar_pfem.')
            or _mod_name == 'torchfem' or _mod_name.startswith('torchfem.')
            or _mod_name == 'pyvista' or _mod_name.startswith('pyvista.')):
        del sys.modules[_mod_name]

import torch
assert torch.cuda.is_available(), 'this cell needs a real GPU'
print('GPU:', torch.cuda.get_device_name(0))

import gc
for _attr in ('last_traceback', 'last_value', 'last_type'):
    if hasattr(sys, _attr):
        delattr(sys, _attr)
gc.collect()
torch.cuda.empty_cache()
print(f'GPU memory at start: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated, '
      f'{torch.cuda.memory_reserved()/1e9:.2f} GB reserved (should be ~0 -- if not, '
      'use Runtime > Restart session before trusting this run)')

from omar_pfem.train_B3_mixed_stress import get_args, train
from omar_pfem.data.data_generate_B3_dataset import DEFAULT_RESOLUTION

device = torch.device('cuda')

R = '/content/drive/MyDrive/pfem_run'
OUTPUT_DIR = f'{R}/b3_mixed_stress_relative'
os.makedirs(OUTPUT_DIR, exist_ok=True)

args = get_args([
    '--n_iters', '50000',
    '--pretrain_iters', '0',
    '--relative_constitutive', '1',
    '--constitutive_eps', '400.0',
    '--w_equilibrium', '1.0',
    '--batch_size', '8',
    '--log_every', '500',
    '--ckpt_every', '5000',
    '--output_dir', OUTPUT_DIR,
    # p_output_scale=250.0 (attempt 2's calibrated value) is this
    # script's own default -- not overridden here.
])
n_elem = (DEFAULT_RESOLUTION[0] - 1) * (DEFAULT_RESOLUTION[1] - 1) * (DEFAULT_RESOLUTION[2] - 1)
print(f'\nTraining at {DEFAULT_RESOLUTION} -> {n_elem} elements, {args.n_iters} iterations, '
      f'batch_size={args.batch_size}, w_constitutive={args.w_constitutive}, '
      f'w_equilibrium={args.w_equilibrium}, p_output_scale={args.p_output_scale}, '
      f'relative_constitutive={bool(args.relative_constitutive)}, '
      f'constitutive_eps={args.constitutive_eps}, checkpoints -> {OUTPUT_DIR}')
print('constitutive is now O(1), not O(10,000) -- watch for a clear DOWNWARD TREND from its '
      'initial ~10-80 range, not just a different oscillation range. If it still shows no '
      'trend, interrupt and report -- this is the fourth independent hypothesis being tested.')

torch.cuda.reset_peak_memory_stats(device)
t0 = time.time()
model = train(args, device)
elapsed_total = time.time() - t0

print(f'\n{"=" * 90}')
print(f'Training finished in {elapsed_total:.1f}s ({elapsed_total / args.n_iters:.4f}s/iteration average).')
print(f'GPU peak memory during training: '
      f'{torch.cuda.max_memory_allocated(device) / 1e6:.1f}MB allocated, '
      f'{torch.cuda.max_memory_reserved(device) / 1e6:.1f}MB reserved.')

# =====================================================================
#  Evaluation: same held-out FEM dataset the baseline DEM checkpoint was
#  evaluated on, so the two sets of numbers are directly comparable.
# =====================================================================
print(f'\n{"=" * 90}')
print('=== Evaluating the final checkpoint against the held-out FEM dataset ===')
CHECKPOINT = f'{OUTPUT_DIR}/checkpoint_{args.n_iters}.pt'
DATASET = f'{R}/b3_dataset_clean_holdout/dataset.h5'
OUT_JSON = f'{OUTPUT_DIR}/qois_{args.n_iters}.json'

sys.argv = [
    'evaluate_B3_mixed_stress.py',
    '--checkpoint', CHECKPOINT,
    '--dataset', DATASET,
    '--out_json', OUT_JSON,
]
from omar_pfem.evaluate_B3_mixed_stress import main as evaluate_main
evaluate_main()

try:
    from omar_pfem.run_manifest import write_manifest
    write_manifest(OUTPUT_DIR, kind='b3_mixed_stress_training',
                    args={'n_iters': args.n_iters, 'pretrain_iters': args.pretrain_iters,
                          'batch_size': args.batch_size,
                          'resolution': DEFAULT_RESOLUTION, 'lr': args.lr,
                          'w_constitutive': args.w_constitutive,
                          'w_equilibrium': args.w_equilibrium,
                          'p_output_scale': args.p_output_scale,
                          'relative_constitutive': bool(args.relative_constitutive),
                          'constitutive_eps': args.constitutive_eps},
                    started_at=_started,
                    results={'elapsed_total_s': elapsed_total,
                             's_per_iter': elapsed_total / args.n_iters},
                    outputs=[CHECKPOINT, OUT_JSON],
                    notes="Fourth real training attempt of the mixed displacement-stress "
                          "formulation (Professor Rabczuk's item 1). Attempts 1 (scale=500), "
                          "2 (scale=250), and 3 (two-stage curriculum) all failed with "
                          "constitutive showing no downward trend. This attempt uses a "
                          "RELATIVE constitutive residual (relative_constitutive=True, "
                          "constitutive_eps=400.0) instead, after measuring the true |P| "
                          "field is extremely heavy-tailed (max ~26x median, likely the "
                          "groove region) -- a plain absolute residual lets those points "
                          "dominate the loss. w_equilibrium re-measured and reduced from "
                          "28000 to 1.0 to match the new loss's natural scale. See "
                          "train_B3_mixed_stress.py's own 'UPDATE 2026-10-22' docstring "
                          "entry for the full diagnosis.")
except Exception as e:
    print(f'[manifest] not recorded: {e}')

print('\nDone. Send this cell\'s full printed output back, especially the "Key comparison '
      'for Professor Rabczuk\'s item 1" line near the end of the evaluation output.')
